Bank Customer Exit Prediction


*   Based on given data using Neural Network Binary Classifier, find out the customer who going to stay or leave
*   Stay: (Churn=0) and leave(churn=1)
- predict the result for new data as well at the end



In [75]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import confusion_matrix, accuracy_score


In [76]:
bank_data = pd.read_csv('/content/bank_churn.csv')
bank_data.head()

,CreditScore,Geography,Gender,Age,Balance,NumOfProducts,EstimatedSalary,Exited
0,600,France,Male,40,60000,1,50000,0
1,700,Spain,Female,45,70000,2,60000,0
2,800,Germany,Female,38,80000,1,70000,1
3,400,France,Male,30,30000,1,40000,1
4,550,Spain,Male,50,40000,2,42000,0


In [77]:
le = LabelEncoder()
bank_data = pd.get_dummies(bank_data, columns=['Geography'])  # OneHotEncoder by using pandas
bank_data['Gender'] = le.fit_transform(bank_data['Gender'])   # LabelEncoder

In [78]:
X = bank_data.drop('Exited', axis=1)
y = bank_data['Exited']

In [79]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [80]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train.shape

(8, 9)

In [81]:
# Model Creation
model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [82]:
# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 16)             │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 305 (1.19 KB)

 Trainable params: 305 (1.19 KB)

 Non-trainable params: 0 (0.00 B)

In [83]:
# Train the model
history_model = model.fit(X_train, y_train, epochs=100,verbose=0, validation_data=(X_test, y_test))

In [84]:
# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print('Test Loss:', loss)
print('Test Accuracy:', accuracy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5000 - loss: 0.9592
Test Loss: 0.9592469930648804
Test Accuracy: 0.5


In [86]:
bank_data.head(5)

,CreditScore,Gender,Age,Balance,NumOfProducts,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,60000,1,50000,0,True,False,False
1,700,0,45,70000,2,60000,0,False,False,True
2,800,0,38,80000,1,70000,1,False,True,False
3,400,1,30,30000,1,40000,1,True,False,False
4,550,1,50,40000,2,42000,0,False,False,True


In [90]:
# Predict a new customer
new_customer = pd.DataFrame({

    'CreditScore': [600],
    'Geography': ['France'],
    'Gender': ['Male'],
    'Age': [30],
    'Balance': [100000],
    'NumOfProducts': [1],
    'EstimatedSalary': [50000]
})
# Encode and scale as before
new_customer['Gender'] = le.transform(new_customer['Gender'])
new_customer = pd.get_dummies(new_customer, columns=['Geography'])

# Add missing dummy columns
for col in X.columns:
    if col not in new_customer.columns:
        new_customer[col] = 0

new_customer = new_customer[X.columns]
new_scaled = scaler.transform(new_customer)

# Predicts
prediction = model.predict(new_scaled)
if prediction[0][0] > 0.5:
  print('Customer will leave')
  print(predict[0][0])
else:
  print('Customer will stay')
  print(prediction[0][0])
print('Prediction:', prediction)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Customer will stay
0.28329378
Prediction: [[0.28329378]]
